[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [LightGBM and XGBoost, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)

# Saving and Loading Boosters &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import json
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb

TERRAINS = ["coastal", "fjord", "inland", "valley", "mountain"]


def station_days(n_stations=400, days=50, seed=2026):
    """Made-up daily minimum temperatures at weather stations in Norway, one row per station and day."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n_stations)
    coast_km = np.minimum(rng.exponential(30, n_stations), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n_stations) + 2.5 * coast_km, 0, 1600)
    terrain = rng.choice(TERRAINS, n_stations, p=[0.2, 0.2, 0.25, 0.2, 0.15])
    has_snow_sensor = rng.uniform(size=n_stations) < 0.7
    offset = rng.normal(0, 0.8, n_stations)
    station = np.repeat(np.arange(n_stations), days)
    n = n_stations * days
    day = rng.integers(1, 366, n)
    cloud = np.clip(rng.beta(1.2, 1.0, n), 0, 1)
    wind = rng.gamma(2.0, 2.0, n)
    season = np.cos(2 * np.pi * (day - 20) / 365)
    lat, elev, coast = latitude[station], elevation[station], coast_km[station]
    snow = np.where(season > 0.2, np.clip(rng.normal(30 * season + 0.03 * elev + 2 * (lat - 58), 15), 0, None), 0.0)
    shelter = pd.Series(terrain[station]).map({"coastal": 0.4, "fjord": 1.0, "inland": 1.2, "valley": 1.7, "mountain": 0.8}).to_numpy()
    mean = 8 - 0.55 * (lat - 58) - 0.006 * elev - 9 * season + 3.5 * np.exp(-coast / 40) * season
    cooling = 7 * (1 - cloud) * np.exp(-wind / 3) * (0.4 + (season + 1) / 2) * shelter
    tmin = mean - 2.5 - cooling - 2.0 * (snow > 10) * (1 - cloud) + offset[station] + rng.normal(0, 1.2, n)
    snow_cm = np.where(has_snow_sensor[station], snow.round(0), np.nan)
    return pd.DataFrame({
        "station": [f"ST{s + 1:04d}" for s in station],
        "latitude": lat.round(2), "elevation_m": elev.round(0).astype(int), "coast_km": coast.round(1),
        "terrain": pd.Categorical(terrain[station], categories=TERRAINS),
        "day_of_year": day, "cloud_cover": cloud.round(2), "wind_ms": wind.round(1), "snow_cm": snow_cm,
        "tmin": tmin.round(1),
    })


FEATURES = ["latitude", "elevation_m", "coast_km", "terrain", "day_of_year", "cloud_cover", "wind_ms", "snow_cm"]
days = station_days()
stations = days["station"].unique()
order = np.random.default_rng(0).permutation(len(stations))
train = days[days["station"].isin(stations[order[:300]])]
valid = days[days["station"].isin(stations[order[300:350]])]
test = days[days["station"].isin(stations[order[350:]])]

scratch = Path("scratch")
scratch.mkdir(exist_ok=True)
print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "| files go in", scratch)

lightgbm 4.6.0 | xgboost 3.4.1 | files go in scratch


The worked examples' two models and files, rebuilt here from the same made-up stations so that each
answer runs on its own.

In [2]:
lightgbm_model = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.1, random_state=0, verbose=-1)
lightgbm_model.fit(train[FEATURES], train["tmin"], eval_set=[(valid[FEATURES], valid["tmin"])],
                   callbacks=[lgb.early_stopping(50, verbose=False)])
lightgbm_file = scratch / "lightgbm_model.txt"
lightgbm_model.booster_.save_model(lightgbm_file)

xgboost_model = xgb.XGBRegressor(n_estimators=1000, learning_rate=0.1, early_stopping_rounds=50, random_state=0)
xgboost_model.fit(train[FEATURES], train["tmin"], eval_set=[(valid[FEATURES], valid["tmin"])], verbose=False)
xgboost_model.save_model(scratch / "xgboost_model.json")

**1.** Save `lightgbm_model`'s booster with `model_to_string()`, load it back with `lgb.Booster(model_str=...)`, and check that it predicts the test stations as the original does.

In [3]:
text = lightgbm_model.booster_.model_to_string()
from_text = lgb.Booster(model_str=text)
print("same predictions:", np.array_equal(from_text.predict(test[FEATURES]), lightgbm_model.predict(test[FEATURES])))

same predictions: True


`model_to_string` returns the model as a string of text, and a `Booster` made from that string predicts every test day as the original does, which lets a model travel without a file.

**2.** Print the lines of LightGBM's file that start with `[num_leaves:` and `[learning_rate:`, the settings the file records near its end.

In [4]:
for line in lightgbm_file.read_text().splitlines():
    if line.startswith(("[num_leaves:", "[learning_rate:")):
        print(line)

[learning_rate: 0.1]
[num_leaves: 31]


Near its end, after the trees, the file lists the settings the model was trained with, each in brackets, among them a learning rate of 0.1 and 31 leaves, the default.

**3.** Train an `XGBClassifier` for frost (`tmin < 0`) with 100 trees, save it to `scratch/frost.json`, load it into a new `XGBClassifier`, and check that `predict_proba` gives the same probabilities.

In [5]:
frost = (train["tmin"] < 0).astype(int)
classifier = xgb.XGBClassifier(n_estimators=100, random_state=0).fit(train[FEATURES], frost)
classifier.save_model(scratch / "frost.json")
classifier_loaded = xgb.XGBClassifier()
classifier_loaded.load_model(scratch / "frost.json")
print("same probabilities:", np.array_equal(classifier_loaded.predict_proba(test[FEATURES]),
                                             classifier.predict_proba(test[FEATURES])))

same probabilities: True


The classifier's file loads into an `XGBClassifier` and gives the same probabilities for every test day.

**4.** Read `scratch/xgboost_model.json` and print how many trees it holds, from `learner['gradient_booster']['model']['gbtree_model_param']`, beside the best round.

In [6]:
learner = json.loads((scratch / "xgboost_model.json").read_text())["learner"]
print("trees:", learner["gradient_booster"]["model"]["gbtree_model_param"]["num_trees"])
print("best round:", learner["attributes"]["best_iteration"])

trees: 127
best round: 76


The file holds 127 trees, every round XGBoost trained, beside a best round of 76, which counting from 0 is the 77th.

In [7]:
shutil.rmtree(scratch)
print("scratch removed:", not scratch.exists())

scratch removed: True


---

&#8592; **Back to:** [Saving and Loading Boosters](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/14-saving-and-loading-boosters.ipynb)  &nbsp;&middot;&nbsp;  [LightGBM and XGBoost, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)
